###2. Quanto tempo dura, em média, a fase de stay period (suspensão de execuções) na prática, até a homologação do Plano de Recuperação Judicial?

**Pergunta:** o art. 6º, §4º, da Lei 11.101/2005 suspende as ações e
execuções contra a empresa em recuperação judicial, a partir do
deferimento do processamento. Na redação original, o prazo era de
180 dias improrrogáveis; a Lei 14.112/2020 permitiu prorrogar uma única vez, por igual período. 

A pergunta mede quanto tempo essa proteção dura de fato — e se o prazo de 180 dias é respeitado na prática.

**Fonte no mart (consumo puro):** `gold_base_analitica` —
`dias_stay_period` (do deferimento até o fim legal da suspensão) e
`dias_stay_limite_superior` (até o fim do processo). 

O cálculo da métrica está no Bloco 4 do notebook 03: a suspensão começa no deferimento do processamento (a mesma data validada na pergunta 1) e termina quando o plano é homologado pelos credores ou quando a empresa tem a falência decretada — o que acontecer primeiro. Este notebook não refaz esses cálculos, mas consome o resultado já pronto.

**Recorte:** classe 129 (pedido principal de RJ), ajuizado a partir de 09/06/2005 (escopo da lei vigente), com processamento identificado e datas coerentes. Agregação Brasil.

**Método:** estatísticas de duração (mediana, média, % acima de 180 dias). 

**Limitação central:** a homologação do plano — o evento que encerraria a suspensão na maioria dos casos — não é digitada pelos tribunais no padrão Datajud. Foi comprovado no QA: nenhum movimento com nome contendo "plano" existe entre ~14,1 milhões de eventos, e nenhum padrão estatístico nas janelas teóricas de decisão ou prorrogação foi constatado (situação validada contra o benchmark da ABJ — Observatório da Insolvência, cap. 7). Essa é uma **limitação do dado, não do método**: sabemos **quando** a
suspensão começou, mas, na maioria dos casos, não observamos
**quando** terminou. Por isso a resposta tem duas camadas:

| Camada | O que mede | Quem tem esse dado |
|---|---|---|
| **Suspensão medida** (fim do stay period = homologação ou falência) | a duração real da proteção | só quem teve a falência decretada |
| **Limite superior** (fim estimado pelo fim do processo) | o máximo que a suspensão pode ter durado | quem tem fim de processo registrado |

#### O que a consulta seguinte mede e como (suspensão medida)

**Nota de escopo:** a homologação do plano — evento que encerraria a
suspensão na maioria dos casos — não é registrada no padrão Datajud. Por isso o fim do stay
period só é observável quando a falência é decretada, e esta consulta
mede a trajetória do fracasso, não a da recuperação bem-sucedida.

**Entrada:** `default.gold_base_analitica`, classe 129, escopo da lei
vigente, com concessão registrada e duração coerente (≥ 0).

A consulta lê três colunas por processo:

| Coluna lida | O que representa |
|---|---|
| `data_processamento_efetivo` | a data do deferimento do processamento — o início da suspensão (a mesma concessão validada na pergunta 1) |
| `dias_stay_period` | dias do deferimento ao fim legal da suspensão (homologação ou decretação de falência, o que ocorrer primeiro) — calculado no Bloco 4 do notebook 03 |
| `ind_stay_period_excedido_180d` | indicador de que a suspensão excedeu o prazo legal de 180 dias |

**Saída — o que ela produz (uma única linha com seis números):**

| Coluna produzida | Como é calculada | O que significa |
|---|---|---|
| `com_processamento` | conta os processos com concessão datada | o denominador de cobertura (2.422) |
| `com_fim_stay` | conta os casos com stay ≥ 0 | quantos têm o fim da suspensão observável (110) |
| `pct_acima_180` | soma do indicador ÷ com_fim_stay × 100 | a fatia que excedeu o prazo legal (97,3%) |
| `media_dias` / `mediana_dias` / `p90_dias` | estatísticas de `dias_stay_period` | a duração típica da suspensão medida e a cauda |

**Em uma frase:** a consulta mede a duração da suspensão nos únicos
casos em que seu fim é observável — as falências decretadas —
produzindo a mediana, a média e a fatia acima do prazo legal que
sustentam a primeira camada da resposta.

In [0]:
%sql
-- Q2 · BRASIL: duração do stay period (consumo do mart)
SELECT
  (SELECT count(*) FROM default.gold_base_analitica
    WHERE classe_codigo = 129
      AND escopo_lei_atual
      AND data_processamento_efetivo IS NOT NULL)        AS com_processamento,
  count(*)                                               AS com_fim_stay,
  round(sum(ind_stay_period_excedido_180d) * 100.0 / count(*), 1) AS pct_acima_180,
  round(avg(dias_stay_period), 1)                        AS media_dias,
  round(percentile_approx(dias_stay_period, 0.5), 1)     AS mediana_dias,
  round(percentile_approx(dias_stay_period, 0.9), 1)     AS p90_dias
FROM default.gold_base_analitica
WHERE classe_codigo = 129
  AND escopo_lei_atual
  AND data_processamento_efetivo IS NOT NULL
  AND dias_stay_period >= 0;

#### O que a próxima consulta mede e como (limite superior da suspensão)

**Entrada:** as mesmas colunas da consulta anterior, agora lendo
`dias_stay_limite_superior` — dias do deferimento até o fim do processo,
o máximo que a suspensão pode ter durado quando seu fim real não é
observável.

**Saída — o que ela produz (uma única linha com cinco números):**

| Coluna produzida | Como é calculada | O que significa |
|---|---|---|
| `com_cota` | conta os casos com limite ≥ 0 | quantos têm ao menos o teto da suspensão estimável (530) |
| `media_cota` / `mediana_cota` / `p90_cota` | estatísticas de `dias_stay_limite_superior` | o piso do "quanto durou no máximo" |
| `pct_cota_acima_180` | casos com cota > 180 ÷ com_cota × 100 | a fatia cuja suspensão, mesmo no cenário mais otimista, excedeu o prazo legal (93,2%) |

**Em uma frase:** a consulta estima o teto da suspensão nos casos sem
fim observável — se até o teto estoura os 180 dias (93,2%), a conclusão
da primeira camada se robustece, não se enfraquece.

In [0]:
%sql
-- Q2 · Cota do stay (limite superior) na população com concessão
SELECT
  count(*)                                                          AS com_cota,
  round(avg(dias_stay_limite_superior), 1)                          AS media_cota,
  round(percentile_approx(dias_stay_limite_superior, 0.5), 1)       AS mediana_cota,
  round(percentile_approx(dias_stay_limite_superior, 0.9), 1)       AS p90_cota,
  round(count(CASE WHEN dias_stay_limite_superior > 180 THEN 1 END)
        * 100.0 / count(*), 1)                                      AS pct_cota_acima_180
FROM default.gold_base_analitica
WHERE classe_codigo = 129 AND escopo_lei_atual
  AND data_processamento_efetivo IS NOT NULL
  AND dias_stay_limite_superior >= 0;

#### Análise da resposta (pergunta 2)

**Cobertura:** dos 2.422 processos com concessão registrada, a
suspensão consegue ser medida até o fim em apenas 110 casos (4,5%) — são exatamente aqueles em que a empresa teve a falência decretada, pois a homologação do plano não é registrada no Datajud. Para outros 530 (21,9%), só é possível saber o **máximo** que a suspensão pode ter durado (até o fim do processo).

**Consegue-se medir apenas os fracassos:** os 110 casos mensuráveis são, por construção, as recuperações que fracassaram. Eles retratam a trajetória do fracasso, não a da recuperação bem-sucedida (por isso não são comparáveis à tramitação total média da pergunta 4, que cobre todos os desfechos).

**Resultados:**

| Camada | N | Mediana | Média | 90% dos casos ficam abaixo de | Acima de 180 dias |
|---|---|---|---|---|---|
| Suspensão medida (falência decretada) | 110 | 1.575 dias (~4,3 anos) | 2.110,5 | 4.354 | 97,3% |
| Limite superior (fim do processo) | 530 | 1.788 dias (~4,9 anos) | 2.051,2 | 4.182 | 93,2% |

**Conclusão:** uma análise confiável da duração do stay period fica prejudicada pela falta de um movimento registrado no Datajud que mostre claramente o fim dessa fase. Uma análise mais aprofundada vai requerer um data mining nos processos completos existente nos tribunais estaduais. 

#### Resposta final (pergunta 2)

A análise confiável da duração do stay period fica prejudicada pela
falta de um movimento registrado no Datajud que mostre claramente o
fim dessa fase: só em 110 casos (as falências decretadas) ela é
mensurável — e ali durou **1.575 dias na mediana**, quase nove vezes o
prazo legal de 180 dias, com 97,3% acima do teto. Nos 530 casos em que
só existe o limite até o fim do processo, o padrão se confirma
(**1.788 dias na mediana**, 93,2% acima do prazo). Aprofundar essa
medida exigirá mineração de dados nos processos completos
(digitados/arquivados) nos tribunais estaduais, fora do padrão
Datajud.

##### Coerência com as demais perguntas
As 110 RJs convoladas com stay period mensurável são subconjunto das 124 convoladas com concessão da pergunta 5 — nos 14 casos restantes, a duração da suspensão não pôde ser computada na gold (campo nulo) e eles ficam fora da métrica, sem implicar incoerência nas datas de concessão ou decretação, que estão registradas. A mediana de 1.575 dias é a mesma citada na decomposição por desfecho da pergunta 4, onde o stay period consome ~94% da vida do processo convolado.